# Encoding Categorical Data in Python
### Ordinal Encoder | Label Encoder | One-Hot Encoder | pd.get_dummies

Machine learning models only understand **numbers**. But real datasets have columns like `City = "Mumbai"` or `Size = "Large"`. These text columns are called **categorical data**.

**Encoding** = converting those text categories into numbers so a model can use them.

In this notebook we go one by one:
1. Ordinal Encoder
2. Label Encoder
3. One-Hot Encoder
4. get_dummies

For each one: definition, syntax, code, output, advantages, disadvantages. At the end there is a full comparison table.

## Step 0: Import libraries and make a small dataset

I am making a small food-delivery style dataset so every encoder is tested on the same data. Then the difference is easy to see.

Two types of categorical columns here:
- **Nominal**: no order (`City`, `Payment`). Mumbai is not "bigger" than Pune.
- **Ordinal**: has a natural order (`Size`: Small < Medium < Large, `Rating`: Poor < Average < Good < Excellent).

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, OneHotEncoder

df = pd.DataFrame({
    "City":    ["Mumbai", "Pune", "Delhi", "Mumbai", "Pune", "Delhi", "Mumbai", "Pune"],
    "Size":    ["Small", "Large", "Medium", "Medium", "Small", "Large", "Large", "Medium"],
    "Rating":  ["Poor", "Good", "Average", "Excellent", "Poor", "Good", "Excellent", "Average"],
    "Payment": ["UPI", "Cash", "Card", "UPI", "Card", "Cash", "UPI", "Cash"],
    "Ordered": ["Yes", "No", "Yes", "Yes", "No", "No", "Yes", "Yes"]   # this is our target (y)
})
df

,City,Size,Rating,Payment,Ordered
0,Mumbai,Small,Poor,UPI,Yes
1,Pune,Large,Good,Cash,No
2,Delhi,Medium,Average,Card,Yes
3,Mumbai,Medium,Excellent,UPI,Yes
4,Pune,Small,Poor,Card,No
5,Delhi,Large,Good,Cash,No
6,Mumbai,Large,Excellent,UPI,Yes
7,Pune,Medium,Average,Cash,Yes


In [2]:
# quick check of the data types, all text columns show up as "object"
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   City     8 non-null      object
 1   Size     8 non-null      object
 2   Rating   8 non-null      object
 3   Payment  8 non-null      object
 4   Ordered  8 non-null      object
dtypes: object(5)
memory usage: 452.0+ bytes


---
# 1. Ordinal Encoder

### Definition
Ordinal Encoder converts categories into numbers **where the order matters**. You tell it the order yourself (Small, Medium, Large) and it gives 0, 1, 2 in that same order.

Use it for: **input features (X) that have a ranking**. Examples: Size, Education level, Rating, T-shirt size.

### Syntax
```python
from sklearn.preprocessing import OrdinalEncoder

enc = OrdinalEncoder(categories=[["low", "mid", "high"]])   # order given by us
enc.fit(df[["column"]])                  # learn the categories
result = enc.transform(df[["column"]])   # convert to numbers
# or both in one step:
result = enc.fit_transform(df[["column"]])
```

Important points:
- Input must be **2D**, so use double brackets `df[["Size"]]`, not `df["Size"]`.
- `categories=[[...]]` is a list of lists, one inner list per column.
- Works on **many columns at once**.

In [3]:
# Ordinal encoding on the Size column with our own order
size_order = ["Small", "Medium", "Large"]

ord_enc = OrdinalEncoder(categories=[size_order])
df_ord = df.copy()
df_ord["Size_encoded"] = ord_enc.fit_transform(df[["Size"]])

df_ord[["Size", "Size_encoded"]]

,Size,Size_encoded
0,Small,0.0
1,Large,2.0
2,Medium,1.0
3,Medium,1.0
4,Small,0.0
5,Large,2.0
6,Large,2.0
7,Medium,1.0


Small became 0, Medium became 1, Large became 2. The order is exactly what we wanted.

What happens if we **don't** give the order? Let's see, this is a common mistake.

In [4]:
# Without giving categories, sklearn sorts ALPHABETICALLY
wrong_enc = OrdinalEncoder()
wrong = wrong_enc.fit_transform(df[["Size"]])

print("Categories it learned:", wrong_enc.categories_)
pd.DataFrame({"Size": df["Size"], "Wrong_encoding": wrong.ravel()})

Categories it learned: [array(['Large', 'Medium', 'Small'], dtype=object)]


,Size,Wrong_encoding
0,Small,2.0
1,Large,0.0
2,Medium,1.0
3,Medium,1.0
4,Small,2.0
5,Large,0.0
6,Large,0.0
7,Medium,1.0


Alphabetical order gives Large=0, Medium=1, Small=2. That is **wrong** because now Large looks smaller than Small. So always pass `categories=` when the order matters.

Now two columns together (Size and Rating), each with its own order:

In [5]:
rating_order = ["Poor", "Average", "Good", "Excellent"]

multi_enc = OrdinalEncoder(categories=[size_order, rating_order])
encoded = multi_enc.fit_transform(df[["Size", "Rating"]])

pd.DataFrame(encoded, columns=["Size_enc", "Rating_enc"]).join(df[["Size", "Rating"]])

,Size_enc,Rating_enc,Size,Rating
0,0.0,0.0,Small,Poor
1,2.0,2.0,Large,Good
2,1.0,1.0,Medium,Average
3,1.0,3.0,Medium,Excellent
4,0.0,0.0,Small,Poor
5,2.0,2.0,Large,Good
6,2.0,3.0,Large,Excellent
7,1.0,1.0,Medium,Average


In [6]:
# Going back to the original text is also possible
multi_enc.inverse_transform(encoded)[:3]

array([['Small', 'Poor'],
       ['Large', 'Good'],
       ['Medium', 'Average']], dtype=object)

### Advantages of Ordinal Encoder
- Keeps the **order** information, so the model knows Large > Medium > Small.
- Does not add extra columns (1 column stays 1 column), so the data stays small.
- Can encode many columns in one go.
- Has `inverse_transform` to get the text back.
- Works well with tree models and also with linear models when the order is real.

### Disadvantages of Ordinal Encoder
- You must write the order manually. If you forget, you get a wrong alphabetical order.
- It assumes equal gaps (0, 1, 2). But is Medium to Large really the same gap as Small to Medium? Not always.
- **Wrong for nominal data.** City = Mumbai(0), Pune(1), Delhi(2) tells the model Delhi > Pune > Mumbai, which is meaningless.
- By default it crashes on new unseen categories (can be handled with `handle_unknown="use_encoded_value"`).

---
# 2. Label Encoder

### Definition
Label Encoder converts each category into a number (0, 1, 2 ...). It is the simplest one. It sorts the categories alphabetically and numbers them.

**Very important:** Label Encoder is designed for the **target column (y)**, not for the input features (X). That is why it only accepts **one column** (1D).

Use it for: target / label column like `Yes/No`, `Spam/Not spam`, `Cat/Dog/Bird`.

### Syntax
```python
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(df["target_column"])   # 1D input, single brackets
le.classes_                                         # shows which number = which label
le.inverse_transform(y_encoded)                     # numbers back to text
```

Notice: here we use **single brackets** `df["Ordered"]`, different from OrdinalEncoder.

In [7]:
le = LabelEncoder()
df_le = df.copy()
df_le["Ordered_encoded"] = le.fit_transform(df["Ordered"])

print("Classes (position = number):", le.classes_)
df_le[["Ordered", "Ordered_encoded"]]

Classes (position = number): ['No' 'Yes']


,Ordered,Ordered_encoded
0,Yes,1
1,No,0
2,Yes,1
3,Yes,1
4,No,0
5,No,0
6,Yes,1
7,Yes,1


`No` became 0 and `Yes` became 1, because N comes before Y alphabetically.

We can also check the mapping and reverse it:

In [8]:
# mapping as a dictionary, easy to read
mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("Mapping:", mapping)

# reverse: numbers -> text
print(le.inverse_transform([0, 1, 1, 0]))

Mapping: {'No': np.int64(0), 'Yes': np.int64(1)}
['No' 'Yes' 'Yes' 'No']


What if we use LabelEncoder on a feature like City? It works, but look at what it does:

In [9]:
city_le = LabelEncoder()
city_codes = city_le.fit_transform(df["City"])

pd.DataFrame({"City": df["City"], "City_label": city_codes})
# Delhi=0, Mumbai=1, Pune=2 -> just alphabetical, the model may think Pune > Mumbai > Delhi

,City,City_label
0,Mumbai,1
1,Pune,2
2,Delhi,0
3,Mumbai,1
4,Pune,2
5,Delhi,0
6,Mumbai,1
7,Pune,2


### Advantages of Label Encoder
- Very simple and fast. Only 2 lines of code.
- No extra columns are created.
- Perfect for the **target variable** (y) in classification.
- `inverse_transform` helps to convert predictions back to real names (for example 1 -> "Yes").
- Works on text and numbers both.

### Disadvantages of Label Encoder
- Only **1 column at a time** (1D data). It can not take a whole DataFrame.
- The numbers are given **alphabetically**, you can not control the order. (Poor/Good/Excellent would not be in the right order.)
- On input features it creates a **fake order** (Delhi < Mumbai < Pune), which can confuse linear models, KNN, SVM etc.
- Not meant for X features, sklearn itself says it is for target labels.

---
# 3. One-Hot Encoder

### Definition
One-Hot Encoder makes a **new column for every category**. The column has `1` if the row belongs to that category and `0` if not. No category gets a bigger number than another, so there is **no fake order**.

Use it for: **nominal** features (no order). Example: City, Payment method, Colour, Gender.

### Syntax
```python
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore", drop=None)
arr = ohe.fit_transform(df[["column"]])    # 2D input, double brackets
ohe.get_feature_names_out()                # names of the new columns
```

Parameters you should know:
| Parameter | What it does |
|---|---|
| `sparse_output=False` | gives a normal numpy array (default True gives a sparse matrix, hard to read) |
| `handle_unknown="ignore"` | if a new category appears later, it gives all zeros instead of an error |
| `drop="first"` | removes the first column to avoid the dummy variable trap |
| `drop=None` | keep all columns (default) |

(In older sklearn versions `sparse_output` was called `sparse`.)

In [10]:
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
city_arr = ohe.fit_transform(df[["City"]])

print("Column names:", ohe.get_feature_names_out())
city_ohe = pd.DataFrame(city_arr, columns=ohe.get_feature_names_out())

# put the original City next to it so we can compare
pd.concat([df[["City"]], city_ohe], axis=1)

Column names: ['City_Delhi' 'City_Mumbai' 'City_Pune']


,City,City_Delhi,City_Mumbai,City_Pune
0,Mumbai,0.0,1.0,0.0
1,Pune,0.0,0.0,1.0
2,Delhi,1.0,0.0,0.0
3,Mumbai,0.0,1.0,0.0
4,Pune,0.0,0.0,1.0
5,Delhi,1.0,0.0,0.0
6,Mumbai,0.0,1.0,0.0
7,Pune,0.0,0.0,1.0


Each city has its own column now. A Mumbai row has `1` only in `City_Mumbai`.

Look closely: if `City_Delhi = 0` and `City_Mumbai = 0`, then the city must be Pune. So the last column is not really needed. It can be predicted from the others. This problem is called the **Dummy Variable Trap** (multicollinearity). It matters for Linear/Logistic Regression. We fix it with `drop="first"`.

In [11]:
ohe_drop = OneHotEncoder(sparse_output=False, drop="first")
arr_drop = ohe_drop.fit_transform(df[["City"]])

pd.DataFrame(arr_drop, columns=ohe_drop.get_feature_names_out())
# Delhi is dropped. Delhi = both columns are 0

,City_Mumbai,City_Pune
0,1.0,0.0
1,0.0,1.0
2,0.0,0.0
3,1.0,0.0
4,0.0,1.0
5,0.0,0.0
6,1.0,0.0
7,0.0,1.0


In [12]:
# One-hot on multiple columns together (City + Payment)
ohe_multi = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
multi_arr = ohe_multi.fit_transform(df[["City", "Payment"]])

pd.DataFrame(multi_arr, columns=ohe_multi.get_feature_names_out())

,City_Delhi,City_Mumbai,City_Pune,Payment_Card,Payment_Cash,Payment_UPI
0,0.0,1.0,0.0,0.0,0.0,1.0
1,0.0,0.0,1.0,0.0,1.0,0.0
2,1.0,0.0,0.0,1.0,0.0,0.0
3,0.0,1.0,0.0,0.0,0.0,1.0
4,0.0,0.0,1.0,1.0,0.0,0.0
5,1.0,0.0,0.0,0.0,1.0,0.0
6,0.0,1.0,0.0,0.0,0.0,1.0
7,0.0,0.0,1.0,0.0,1.0,0.0


In [13]:
# handle_unknown="ignore" in action: a city that was never seen in training
new_data = pd.DataFrame({"City": ["Nagpur"]})
print(ohe.transform(new_data))   # all zeros, no crash

[[0. 0. 0.]]


### Advantages of One-Hot Encoder
- **No fake order.** Every category is treated equally. Best choice for nominal data.
- Works well with Linear Regression, Logistic Regression, KNN, SVM, Neural Networks.
- Can handle new unseen categories safely with `handle_unknown="ignore"`.
- It is a sklearn transformer, so it fits inside a **Pipeline** and `ColumnTransformer` (important for real projects).
- Remembers what it learned from training data, so train and test always get the same columns.

### Disadvantages of One-Hot Encoder
- **Many columns.** A column with 1000 unique values (like Pincode) creates 1000 new columns, so the data becomes big and slow. This is the "curse of dimensionality".
- Dummy variable trap if you keep all columns in linear models (use `drop="first"`).
- Output is a numpy array, so you need extra code to turn it back into a DataFrame with column names.
- Not good for ordinal data, because it throws away the order.

---
# 4. get_dummies (pandas)

### Definition
`pd.get_dummies()` does the same job as One-Hot Encoding, but it comes from **pandas**, not sklearn. You give it a DataFrame and it directly returns a DataFrame with the new 0/1 columns, with proper names. It is the quickest way for analysis and EDA.

Use it for: quick exploration, small projects, data analysis, when you do not need a Pipeline.

### Syntax
```python
pd.get_dummies(df, columns=["col1", "col2"], drop_first=False, dtype=int, prefix=None)
```

| Parameter | What it does |
|---|---|
| `columns=[...]` | which columns to encode (if not given, all text columns are encoded) |
| `drop_first=True` | removes the first category column (avoids dummy variable trap) |
| `dtype=int` | gives 0/1. Newer pandas gives True/False by default, so set this |
| `prefix=` | custom prefix for new column names |

In [14]:
# get_dummies on one column
pd.get_dummies(df["City"], dtype=int)

,Delhi,Mumbai,Pune
0,0,1,0
1,0,0,1
2,1,0,0
3,0,1,0
4,0,0,1
5,1,0,0
6,0,1,0
7,0,0,1


In [15]:
# on the full DataFrame, selected columns only. Other columns stay as they are.
dummies_df = pd.get_dummies(df, columns=["City", "Payment"], dtype=int)
dummies_df

,Size,Rating,Ordered,City_Delhi,City_Mumbai,City_Pune,Payment_Card,Payment_Cash,Payment_UPI
0,Small,Poor,Yes,0,1,0,0,0,1
1,Large,Good,No,0,0,1,0,1,0
2,Medium,Average,Yes,1,0,0,1,0,0
3,Medium,Excellent,Yes,0,1,0,0,0,1
4,Small,Poor,No,0,0,1,1,0,0
5,Large,Good,No,1,0,0,0,1,0
6,Large,Excellent,Yes,0,1,0,0,0,1
7,Medium,Average,Yes,0,0,1,0,1,0


In [16]:
# drop_first=True
pd.get_dummies(df, columns=["City", "Payment"], drop_first=True, dtype=int)

,Size,Rating,Ordered,City_Mumbai,City_Pune,Payment_Cash,Payment_UPI
0,Small,Poor,Yes,1,0,0,1
1,Large,Good,No,0,1,1,0
2,Medium,Average,Yes,0,0,0,0
3,Medium,Excellent,Yes,1,0,0,1
4,Small,Poor,No,0,1,0,0
5,Large,Good,No,0,0,1,0
6,Large,Excellent,Yes,1,0,0,1
7,Medium,Average,Yes,0,1,1,0


In [17]:
# problem with get_dummies: train/test mismatch
train = pd.DataFrame({"City": ["Mumbai", "Pune", "Delhi"]})
test  = pd.DataFrame({"City": ["Mumbai", "Nagpur"]})

train_d = pd.get_dummies(train, dtype=int)
test_d  = pd.get_dummies(test, dtype=int)

print("Train columns:", list(train_d.columns))
print("Test columns :", list(test_d.columns))
# Different columns! A model trained on train_d will fail on test_d.

Train columns: ['City_Delhi', 'City_Mumbai', 'City_Pune']
Test columns : ['City_Mumbai', 'City_Nagpur']


In [18]:
# manual fix: align test to the train columns (missing ones filled with 0)
test_fixed = test_d.reindex(columns=train_d.columns, fill_value=0)
test_fixed

,City_Delhi,City_Mumbai,City_Pune
0,0,1,0
1,0,0,0


The mismatch above is the biggest weakness of `get_dummies`. OneHotEncoder does not have this problem because it remembers the training categories.

### Advantages of get_dummies
- Very easy, one line of code.
- Returns a **DataFrame with column names** directly, no extra work.
- Can encode the whole DataFrame and keeps the other columns untouched.
- Great for quick EDA, visualisation and small projects.
- No sklearn import needed.

### Disadvantages of get_dummies
- **Does not remember** the categories. Train and test can end up with different columns.
- Can not be used inside an sklearn Pipeline (no `fit` / `transform`).
- A new category in production data will not be handled automatically.
- Same problems as One-Hot: too many columns for high-cardinality features, dummy variable trap.
- Newer pandas returns `True/False` instead of `1/0` unless you set `dtype=int`.

---
# 5. Comparison of all four encoders

### Table 1: Same data, different encoding

Let's put all four outputs side by side so the difference is visible.

In [19]:
# Ordinal on Size, Label on City, One-Hot on City, get_dummies on City
compare = pd.DataFrame({"City": df["City"], "Size": df["Size"]})

compare["Size_Ordinal"] = OrdinalEncoder(categories=[size_order]).fit_transform(df[["Size"]]).ravel()
compare["City_Label"]   = LabelEncoder().fit_transform(df["City"])

ohe_c = OneHotEncoder(sparse_output=False)
ohe_part = pd.DataFrame(ohe_c.fit_transform(df[["City"]]),
                        columns=["OHE_" + c.replace("City_", "") for c in ohe_c.get_feature_names_out()])

gd_part = pd.get_dummies(df["City"], prefix="GD", dtype=int)

pd.concat([compare, ohe_part, gd_part], axis=1)

,City,Size,Size_Ordinal,City_Label,OHE_Delhi,OHE_Mumbai,OHE_Pune,GD_Delhi,GD_Mumbai,GD_Pune
0,Mumbai,Small,0.0,1,0.0,1.0,0.0,0,1,0
1,Pune,Large,2.0,2,0.0,0.0,1.0,0,0,1
2,Delhi,Medium,1.0,0,1.0,0.0,0.0,1,0,0
3,Mumbai,Medium,1.0,1,0.0,1.0,0.0,0,1,0
4,Pune,Small,0.0,2,0.0,0.0,1.0,0,0,1
5,Delhi,Large,2.0,0,1.0,0.0,0.0,1,0,0
6,Mumbai,Large,2.0,1,0.0,1.0,0.0,0,1,0
7,Pune,Medium,1.0,2,0.0,0.0,1.0,0,0,1


### Table 2: Feature comparison

| Point | Ordinal Encoder | Label Encoder | One-Hot Encoder | get_dummies |
|---|---|---|---|---|
| **Library** | sklearn | sklearn | sklearn | pandas |
| **Best for** | Ordered categories (Low < Mid < High) | Target column (y) | Nominal features (no order) | Nominal features, quick analysis |
| **Input shape** | 2D `df[["col"]]` | 1D `df["col"]` | 2D `df[["col"]]` | DataFrame or Series |
| **Multiple columns at once** | Yes | No | Yes | Yes |
| **Output** | numpy array, 1 col per input col | numpy array, 1D | numpy array, 1 col per category | DataFrame, 1 col per category |
| **Number of new columns** | Same as input | 1 | = number of categories | = number of categories |
| **Order of numbers** | You decide (`categories=`) | Alphabetical, fixed | No order (0/1 flags) | No order (0/1 flags) |
| **Creates fake order?** | No, if order is set right | Yes, on features | No | No |
| **Handles unseen category** | Optional (`handle_unknown`) | No (error) | Yes (`handle_unknown="ignore"`) | No |
| **Remembers training data** | Yes | Yes | Yes | No |
| **Works in Pipeline** | Yes | No (target only) | Yes | No |
| **Reverse to text** | `inverse_transform` | `inverse_transform` | `inverse_transform` | Manual |
| **Memory usage** | Low | Low | High if many categories | High if many categories |
| **Speed / ease** | Easy | Easiest | Medium | Easiest |

### Table 3: Same column, what the model sees

| City | Label Encoder | One-Hot (3 columns) |
|---|---|---|
| Delhi | 0 | 1, 0, 0 |
| Mumbai | 1 | 0, 1, 0 |
| Pune | 2 | 0, 0, 1 |

With Label encoding the model thinks Pune (2) is "twice" Mumbai (1). With One-Hot every city is equal.

### Which one should I use? (quick decision guide)

1. **Is it the target column (y)?** -> Label Encoder
2. **Does the category have a real order** (Small < Medium < Large)? -> Ordinal Encoder
3. **No order, few categories** (City, Payment)?
   - Building a real model or a pipeline -> One-Hot Encoder
   - Quick analysis / practice / EDA -> get_dummies
4. **No order, but hundreds of categories** (Pincode, Product ID)? -> One-Hot will explode the columns. Look at other methods like Target Encoding or Frequency Encoding (not covered here).

### Which model needs which?
- **Tree models** (Decision Tree, Random Forest, XGBoost): can survive with ordinal/label numbers because they split on thresholds, but One-Hot is still safer for nominal data.
- **Linear / Logistic Regression, KNN, SVM, Neural Network**: need One-Hot for nominal data, because they treat numbers as real distances.

---
# 6. Real-life example: all encoders in one pipeline

In a real project we encode different columns with different encoders together. `ColumnTransformer` does this. This is the "proper" way and it avoids the train/test mismatch problem.

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

X = df[["City", "Size", "Rating", "Payment"]]
y = LabelEncoder().fit_transform(df["Ordered"])      # target -> Label Encoder

preprocess = ColumnTransformer([
    ("ordinal", OrdinalEncoder(categories=[size_order, rating_order]), ["Size", "Rating"]),
    ("onehot",  OneHotEncoder(handle_unknown="ignore", drop="first"), ["City", "Payment"]),
])

model = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression())
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
model.fit(X_train, y_train)
print("Accuracy:", model.score(X_test, y_test))

Accuracy: 0.0


In [21]:
# see the final column names after encoding
model.named_steps["prep"].get_feature_names_out()

array(['ordinal__Size', 'ordinal__Rating', 'onehot__City_Mumbai',
       'onehot__City_Pune', 'onehot__Payment_Cash', 'onehot__Payment_UPI'],
      dtype=object)

Here:
- `Size`, `Rating` -> **Ordinal Encoder** (they have order)
- `City`, `Payment` -> **One-Hot Encoder** (no order)
- `Ordered` (target) -> **Label Encoder**

The dataset has only 8 rows, so the accuracy number is not meaningful. The point here is the **structure**, not the score.

---
# Summary

| Encoder | One line to remember |
|---|---|
| Ordinal Encoder | Numbers **with order**, for features like Size / Rating |
| Label Encoder | Numbers for the **target (y)** only |
| One-Hot Encoder | New 0/1 column per category, for **nominal features**, best for pipelines |
| get_dummies | Pandas version of one-hot, **quick and easy**, but forgets training categories |

### Common mistakes to avoid
- Using LabelEncoder on feature columns (X).
- Forgetting `categories=` in OrdinalEncoder, which gives alphabetical order.
- Using single brackets `df["col"]` where 2D `df[["col"]]` is needed (error: "Expected 2D array").
- Keeping all one-hot columns in Linear / Logistic Regression (dummy variable trap).
- Using `get_dummies` separately on train and test data.

### Practice task
Take any dataset from Kaggle (Titanic is good). Encode `Sex` and `Embarked` with One-Hot, `Pclass` with Ordinal, `Survived` with Label Encoder, then train a model with a Pipeline.